### Hi everyone!) I’m trying to learn using publicly available notebooks and ideas from GitHub. In this notebook, I sort the publicly available notebooks by speed, and starting with the weakest ones and moving up to the leaders, I run them and study how they play so that, in the end, I can create something of my own based on your solutions.
### I’m learning with DEEPSEEK. I have 3 dialogues in it: in one I’m studying your work, in another I’m looking at ideas from GitHub, and in the third I’m reading the discussions!))

# Kaggriculture: Game Map and Work Plan


## What this notebook is

This is a **game map, not an agent**. There is no bot code here — everything is collected in one place:

- all rules and mechanics of Kaggriculture,
- all engine constants (prices, yields, market curves),
- tables for crops, animals, shops, land, and hiring,
- season phase strategy and common beginner mistakes,
- the work plan for our agent.

**Why:** so that anyone (including myself a month from now) can open this, understand the game in 20 minutes, and see what we are doing and why.

---

## What Kaggriculture is

Two players. Each has a 10×10 farm. Only the NW quadrant 5×5 is unlocked at the start. Starting bank — **$3,000**. The season lasts **720 turns = 30 days × 24 turns**.

**Whoever ends with the most coins in the bank wins.** The shed, crops on the farm, and animals do not count. Only coins.

Each turn:
- the **farmer** performs 1 action,
- **each hired hand** performs 1 action,
- up to **10 market orders** can be issued.

---

## Contents

1. **Mechanics** — what you can actually do
2. **Crops** — all parameters + profit per tile-day
3. **Animals** — all parameters + payback time
4. **Market** — prices, curves, what crashes
5. **Town shops** — who buys what
6. **Land and hiring** — quadrants, Fibonacci, shed
7. **Life cycles** — plants and animals
8. **Season phase strategy** — days 1-5, 6-15, 16-24, 25-30
9. **Common beginner mistakes**
10. **Agent work plan** — what we do next

---

## Key facts in 30 seconds

| Item | Value |
|---|---|
| Season length | 720 turns (30 days × 24) |
| Win condition | most coins in the bank at the end |
| Best crop | Wheat (safe) / Carrot (balanced) |
| Best animal | Goose (fast) / Cow (profitable) |
| Shed | 100 items, overflow is lost |
| Hiring | Fibonacci: 1, 1, 2, 3, 5, 8, 13... per day |
| Land | NE $1k → SW $2k → SE $4k (fixed order) |
| Price floor | $1 |

---

## What this notebook does NOT cover

- Agent code (that lives in `main.py`).
- Analysis of specific replays (separate notebook).
- RL/BC training (not our path — see the plan below).

---

## Agent work plan

### Current state

- Agent: a copy of the public **V45** (first-turn wheat round trip) with our layers on top.
- LB peak: ~2700.
- Currently hovering around: ~2500 (the drop is the Elo race condition, not a weakness in the bot).
- The final result is computed with **Bradley-Terry** over all episodes, not the live Elo.

### What we already know

1. **The live LB lies** — the drop 2700 → 2500 → 2000 is partly the concurrent-episode bug.
2. **The final BT scores by win/loss pairs**, not by coin margin.
3. **Copying V45** gives a stable baseline, but no growth.
4. **RL without a GPU** is not our path — too slow.
5. **Our path:** layers on top of the copy, justified by replay analysis.

### Step by step

**Step 1. Analyze leader replays (top-10, rating 3000+)**
- Download 5-10 replays.
- Write a parser → DataFrame (money, actions, crops, animals, prices).
- Build metrics: cash curve, tile usage, action counts, market timing.
- Find 3-5 divergence points: where the leader does X and we do Y.

**Step 2. Hypotheses**
- For each divergence, state a hypothesis: "the leader does X because Y".
- Verify across several replays, not just one.

**Step 3. Layers on top of our agent**
- Each hypothesis becomes a minimal 10-50 line layer.
- Each layer can be switched on/off via `settings`.
- A failure inside a layer falls back to the base command.

**Step 4. Local testing**
- Harness: 64 seeds × 5 opponents (V39, V45, Thomas 2945, Shop Router, random).
- Metric: win rate + mean margin.
- Paired test: same seeds for A and B versions.
- Threshold: change must exceed 2 standard errors.

**Step 5. Submission**
- No more than 1 submission per day.
- Only if the local win rate improved.
- Keep the last 2 slots as a "final pair" for BT.

**Step 6. Freeze**
- Two days before the deadline (September 28) — do not touch anything.
- Watch the BT leaderboard, not the live Elo.

### What we do NOT do

- We do not write an agent from scratch.
- We do not train RL without a GPU.
- We do not chase the live LB peak.
- We do not copy a new bot every day — we stabilize what we have.

---

## Links

- [thomastschinkel: The 2945 Farm](https://www.kaggle.com/code/thomastschinkel/the-2945-farm-96-vs-the-top-10-public-bots/notebook)
- [Ahmed Berat Özer V45](https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v45-first-turn-wheat-round-trip/notebook)

---

## Notes

- Mechanics fully covered — sections 1-9.
- Next: leader replays, find divergence points, write our layers.
- This notebook is updated as we go.

In [ ]:
# Установим и импортируем движок Kaggriculture
!pip install -q -U kaggle-environments

import inspect
from kaggle_environments import make
import kaggle_environments.envs.kaggriculture.kaggriculture as kg

# Какие константы вообще есть в движке
names = [n for n in dir(kg) if n.isupper()]
print("Константы в движке:")
for n in names:
    print(" ", n)

In [ ]:
import pandas as pd
import kaggle_environments.envs.kaggriculture.kaggriculture as kg

# ============ КУЛЬТУРЫ ============
crops_rows = []
for name, c in kg.CROPS.items():
    base_price = kg.MARKET_PARAMS[name]["base"]
    if c["ongoing"]:
        days_occupied = c["first_yield_day"] + c["interval"] * (c["max_yield"] - 1) + 1
    else:
        days_occupied = c["max_yield_day"]
    revenue = c["max_yield"] * base_price
    profit = revenue - c["seed"]
    crops_rows.append({
        "Культура": name,
        "Тип": "Постоянная" if c["ongoing"] else "Одноразовая",
        "Семя $": c["seed"],
        "Цена $": base_price,
        "1-й урожай (дн)": c["first_yield_day"],
        "Макс урожай (дн)": c["max_yield_day"],
        "Макс штук": c["max_yield"],
        "Интервал": c["interval"] if c["interval"] else "—",
        "Дней занято": days_occupied,
        "Выручка $": revenue,
        "Прибыль $": profit,
        "Прибыль/день $": round(profit / days_occupied, 1),
    })

crops_df = pd.DataFrame(crops_rows).sort_values("Прибыль/день $", ascending=False).reset_index(drop=True)

display(
    crops_df.style
        .background_gradient(subset=["Прибыль/день $"], cmap="Greens")
        .background_gradient(subset=["Прибыль $"], cmap="Blues")
        .format({"Прибыль/день $": "${:.1f}"})
        .set_caption("🌾 КУЛЬТУРЫ — на одну клетку, без удобрения")
        .set_table_styles([{
            "selector": "caption",
            "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "8px")]
        }])
        .hide(axis="index")
)

# ============ ЖИВОТНЫЕ ============
WHEAT_PRICE = kg.MARKET_PARAMS["WHEAT"]["base"]

animal_rows = []
for name, a in kg.ANIMALS.items():
    product = a["product"]
    product_price = kg.MARKET_PARAMS[product]["base"]
    prod_per_day = 1.0 / a["interval"]
    revenue_per_day = prod_per_day * product_price
    feed_cost = WHEAT_PRICE
    profit_per_day = revenue_per_day - feed_cost
    payback = round(a["cost"] / profit_per_day, 1) if profit_per_day > 0 else "—"
    animal_rows.append({
        "Животное": name,
        "Цена $": a["cost"],
        "Постройка": a["structure"],
        "Продукт": product,
        "Цена прод $": product_price,
        "1-й прод (дн)": a["first_yield_day"],
        "Каждые N дн": a["interval"],
        "Штук/день": round(prod_per_day, 2),
        "Выручка/день $": round(revenue_per_day, 1),
        "Корм/день $": feed_cost,
        "Прибыль/день $": round(profit_per_day, 1),
        "Окупаемость (дн)": payback,
        "Макс в клетке": a["max_held"],
    })

animals_df = pd.DataFrame(animal_rows).sort_values("Прибыль/день $", ascending=False).reset_index(drop=True)

display(
    animals_df.style
        .background_gradient(subset=["Прибыль/день $"], cmap="Oranges")
        .background_gradient(subset=["Окупаемость (дн)"], cmap="Reds_r")
        .format({"Прибыль/день $": "${:.1f}", "Выручка/день $": "${:.1f}"})
        .set_caption("🐄 ЖИВОТНЫЕ — прибыль за вычетом корма (1 пшеница = $25/день)")
        .set_table_styles([{
            "selector": "caption",
            "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "8px")]
        }])
        .hide(axis="index")
)

In [ ]:
import pandas as pd
import kaggle_environments.envs.kaggriculture.kaggriculture as kg

# ============ РЫНОК — цены и кривые ============
def describe_target(t):
    if t >= 1.5: return "🔥 обваливается в $1"
    if t >= 1.0: return "⚠️ падает сильно"
    if t >= 0.5: return "падает умеренно"
    return "устойчивая"

market_rows = []
for name, p in kg.MARKET_PARAMS.items():
    market_rows.append({
        "Продукт": name,
        "База $": p["base"],
        "T (порог)": p["T"],
        "Кривая ↓": p["below_func"],
        "Удар ↓": p["below_target"],
        "Кривая ↑": p["above_func"],
        "Рост ↑": p["above_target"],
        "Характер": describe_target(p["above_target"]),
    })

market_df = pd.DataFrame(market_rows).sort_values("База $", ascending=False).reset_index(drop=True)

display(
    market_df.style
        .background_gradient(subset=["База $"], cmap="Greens")
        .background_gradient(subset=["Удар ↓"], cmap="Reds")
        .background_gradient(subset=["Рост ↑"], cmap="Blues")
        .format({"База $": "${:.0f}", "Удар ↓": "{:.2f}", "Рост ↑": "{:.2f}"})
        .set_caption("💰 РЫНОК — цены и кривые (I0=10000, пол $1)")
        .set_table_styles([{
            "selector": "caption",
            "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "8px")]
        }])
        .hide(axis="index")
)

# ============ МАГАЗИНЫ ============
shop_rows = []
for shop, products in kg.SHOPS.items():
    shop_rows.append({
        "Магазин": shop,
        "Что покупает": " + ".join(products),
        "Продуктов": len(products),
        "Множитель": "×2" if len(products) == 1 else "×1",
    })

shops_df = pd.DataFrame(shop_rows).sort_values("Продуктов").reset_index(drop=True)

display(
    shops_df.style
        .background_gradient(subset=["Продуктов"], cmap="Purples")
        .set_caption("🏪 МАГАЗИНЫ — кто что покупает (открывается каждые 3 дня, ест каждые 4 хода)")
        .set_table_styles([{
            "selector": "caption",
            "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "8px")]
        }])
        .hide(axis="index")
)

# ============ ГОРОДСКОЙ ЦЕНТР ============
print()
print("⭐ ГОРОДСКОЙ ЦЕНТР покупает по 1 шт. ВСЕХ продуктов (кроме удобрения) раз в день:")
print("   " + ", ".join(kg.TOWN_CENTER_PRODUCTS))
print(f"   → {len(kg.TOWN_CENTER_PRODUCTS)} продуктов × 30 дней = {len(kg.TOWN_CENTER_PRODUCTS)*30} шт. за сезон")
print()
print("💡 КЛЮЧЕВЫЕ ФАКТЫ:")
print("   • MELON — ни один магазин не покупает. Только горцентр: 30 шт. за сезон.")
print("     При продаже >158 шт. цена падает до $1.")
print("   • WOOL — только YARN_STORE. Если его нет — цена обваливается в $1.")
print("   • FERTILIZER — НИКТО не покупает, кроме игроков (BUY_PRODUCT).")
print("   • Дефицит поднимает цену: CARROT при -1000 = $531 (×15), TOMATO при -1000 = $3252 (×54)")

In [ ]:
import pandas as pd
import kaggle_environments.envs.kaggriculture.kaggriculture as kg

# ============ ЗЕМЛЯ ============
land_rows = []
cum = 0
for i, (quad, price) in enumerate(zip(kg.LAND_ORDER, kg.LAND_PRICES)):
    cum += price
    land_rows.append({
        "Порядок": i + 1,
        "Квадрант": quad,
        "Цена $": price,
        "Накопительно $": cum,
        "Клеток всего": 25 * (i + 2),
    })

land_df = pd.DataFrame(land_rows)

display(
    land_df.style
        .background_gradient(subset=["Цена $"], cmap="YlOrBr")
        .format({"Цена $": "${:,.0f}", "Накопительно $": "${:,.0f}"})
        .set_caption("🗺️ ЗЕМЛЯ — старт NW (25 клеток), докупка по фиксированному порядку")
        .set_table_styles([{
            "selector": "caption",
            "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "8px")]
        }])
        .hide(axis="index")
)

# ============ НАЙМ (ФИБОНАЧЧИ) ============
fibs = [1, 1]
for _ in range(8):
    fibs.append(fibs[-1] + fibs[-2])

hire_rows = []
total = 0
for n in range(1, 11):
    total += fibs[n - 1]
    hire_rows.append({
        "Найм №": n,
        "Цена $": fibs[n - 1],
        "Всего потрачено $": total,
        "Рабочих в день": n,
        "Действий/день": n * 24,
        "Цена за действие $": round(total / (n * 24), 3),
    })

hire_df = pd.DataFrame(hire_rows)

display(
    hire_df.style
        .background_gradient(subset=["Цена $"], cmap="Oranges")
        .background_gradient(subset=["Цена за действие $"], cmap="Greens_r")
        .format({"Цена $": "${:,.0f}", "Всего потрачено $": "${:,.0f}",
                 "Цена за действие $": "${:.3f}"})
        .set_caption("👷 НАЙМ — числа Фибоначчи, сбрасывается каждый день")
        .set_table_styles([{
            "selector": "caption",
            "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "8px")]
        }])
        .hide(axis="index")
)

# ============ СТАРТ ============
print()
print("=" * 100)
print("🚜 СТАРТОВЫЕ УСЛОВИЯ")
print("=" * 100)
print(f"  Доска:              10×10 (4 квадранта 5×5)")
print(f"  Стартовая земля:    NW (25 клеток)")
print(f"  Стартовые деньги:   $3,000")
print(f"  Фермер:             1 штука, спавн у сарая")
print(f"  Рабочих:            до N, найм через HIRE, фибоначчи")
print(f"  Сарай:              100 предметов (без семян), переполнение = потеря")
print(f"  Длина сезона:       720 ходов = 30 дней × 24 хода")
print(f"  Победа:             больше всего денег в банке в конце")
print(f"  Сарай-адъяцентные:  (4,4), (5,4), (4,5), (5,5)")
print(f"  Пол цены:           ${kg.PRICE_FLOOR}")
print(f"  I0 (старт инвентаря рынка): {kg.MARKET_I0:,}")

# ============ ДЕНЬ ============
print()
print("=" * 100)
print("⏰ ЧТО ПРОИСХОДИТ КАЖДЫЙ ДЕНЬ")
print("=" * 100)
print("  Каждый ход:")
print("    • Фермер делает 1 действие")
print("    • Каждый рабочий делает 1 действие")
print("    • Можно отдать до 10 market-ордеров")
print()
print("  Каждые 4 хода:  каждый открытый магазин съедает своё меню")
print("  Каждый день:    городской центр съедает 1 шт. каждого продукта")
print("  В конце дня:")
print("    • Рабочие исчезают, их карман выкладывается в сарай")
print("    • Фермер телепортируется к сараю на утро")
print("    • Сброс флагов watered_today / fed_today")
print("    • Если сарай > 100 предметов — лишнее уничтожается")
print("    • Растения без полива 2 дня → сорняк")
print("    • Животные без корма 2 дня → сбежали")

print()
print("=" * 100)
print("🌱 ЖИЗНЕННЫЙ ЦИКЛ РАСТЕНИЯ")
print("=" * 100)
print("  1. PLANT — посадил (день 0). ВАЖНО: этот день уже считается неполитым.")
print("  2. WATER каждый день — иначе через 2 дня сорняк.")
print("  3. Окно бонуса: со дня ceil(max_yield_day/2) до max_yield_day.")
print("     В этом окне каждый полив даёт +1 к урожаю (или +2 с удобрением).")
print("  4. HARVEST — собрал. Одноразовые убираются, постоянные остаются.")
print("  5. Decay — после max_lifespan_step теряет 1 ед. каждые 2 хода → сорняк.")

print()
print("=" * 100)
print("🐮 ЖИЗНЕННЫЙ ЦИКЛ ЖИВОТНОГО")
print("=" * 100)
print("  1. BUILD_COOP / BUILD_PASTURE — построил (1 действие).")
print("  2. PLACE ANIMAL — поставил (1 действие).")
print("  3. FEED каждый день (1 пшеница) — иначе через 2 дня сбежал.")
print("  4. CARE каждый день — банкует +1 к следующему продукту.")
print("  5. COLLECT_FERTILIZER — 1 удобрение в день бесплатно.")
print("  6. HARVEST продукт — когда появился (по расписанию).")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import kaggle_environments.envs.kaggriculture.kaggriculture as kg

def shape(func, x, T=None):
    x = max(0, x)
    if func == "linear": return x
    if func == "sq": return x * x
    if func == "sqrt": return math.sqrt(x)
    if func == "log": return math.log(1 + x)
    if func == "log10": return math.log10(1 + x)
    if func == "hinge":
        if not T or T <= 0: return x
        u = x / T
        return u + 8 * max(0, u - 1) ** 2
    return x

def market_price(item, inv):
    p = kg.MARKET_PARAMS[item]
    base, I0, T = p["base"], p["I0"], p["T"]
    if inv < I0:
        f = p["below_func"]
        amp = p["below_target"] * base / shape(f, T, T)
        price = base + amp * shape(f, I0 - inv, T)
    else:
        f = p["above_func"]
        amp = p["above_target"] * base / shape(f, T, T)
        price = base - amp * shape(f, inv - I0, T)
    return max(kg.PRICE_FLOOR, round(price))

items = list(kg.MARKET_PARAMS.keys())
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, item in zip(axes.ravel(), items):
    # от -2000 (дефицит) до +2000 (переизбыток) относительно I0
    invs = np.arange(kg.MARKET_I0 - 2000, kg.MARKET_I0 + 2001, 50)
    prices = [market_price(item, int(i)) for i in invs]
    base = kg.MARKET_PARAMS[item]["base"]
    ax.plot(invs - kg.MARKET_I0, prices, color="tab:blue", lw=2)
    ax.axhline(base, color="gray", ls="--", lw=1, alpha=0.6)
    ax.axvline(0, color="black", ls=":", lw=1, alpha=0.4)
    ax.set_title(f"{item}  (база ${base})", fontweight="bold")
    ax.set_xlabel("отклонение от I0")
    ax.set_ylabel("цена $")
    ax.grid(alpha=0.3)
plt.suptitle("Как цена реагирует: влево — дефицит (дорого), вправо — переизбыток (дёшево)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Для каждого продукта: сколько единиц надо продать от I0, чтобы цена упала до $1
floor_rows = []
for item in kg.MARKET_PARAMS:
    base = kg.MARKET_PARAMS[item]["base"]
    # ищем сколько единиц выше I0 нужно, чтобы цена достигла пола
    for units in range(0, 5000, 10):
        if market_price(item, kg.MARKET_I0 + units) <= 1:
            break
    floor_rows.append({
        "Продукт": item,
        "База $": base,
        "До $1 (шт)": units,
        "Магазины покупают": ", ".join([s for s, prods in kg.SHOPS.items() if item in prods]) or "нет",
        "Горцентр": "✅" if item in kg.TOWN_CENTER_PRODUCTS else "❌",
    })

floor_df = pd.DataFrame(floor_rows).sort_values("До $1 (шт)")

display(
    floor_df.style
        .background_gradient(subset=["До $1 (шт)"], cmap="Reds_r")
        .format({"База $": "${:.0f}"})
        .set_caption("💥 СКОЛЬКО НАДО ПРОДАТЬ, ЧТОБЫ ОБВАЛИТЬ ЦЕНУ В $1")
        .set_table_styles([{
            "selector": "caption",
            "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "8px")]
        }])
        .hide(axis="index")
)

In [ ]:
print("=" * 100)
print("📅 СТРАТЕГИЯ ПО ФАЗАМ СЕЗОНА (30 дней)")
print("=" * 100)
print()
print("🟢 ДНИ 1–5: РАЗГОН (у тебя только NW, 25 клеток, $3000)")
print("   • Покупай семена пшеницы/моркови — они дешёвые и быстрые (2-3 дня).")
print("   • Сажай по максимуму, поливай каждый день.")
print("   • Пшеница = и товар, и корм для животных. Копи её.")
print("   • Продавай понемногу, чтобы хватало на новые семена.")
print("   • Начинай строить курятник (BUILD_COOP) — 1 действие.")
print()
print("🟡 ДНИ 6–15: РАСШИРЕНИЕ")
print("   • Купи NE ($1000). У тебя 50 клеток.")
print("   • Нанимай 2–4 рабочих (фибоначчи — дёшево).")
print("   • Добавляй гусят ($300) — они дают $50 в день каждый.")
print("   • Продавай пшеницу/яйца стабильно, копи на большой скачок.")
print("   • Строй пастбище (BUILD_PASTURE) для коров/овец.")
print()
print("🟠 ДНИ 16–24: ПРОИЗВОДСТВО")
print("   • Купи SW ($2000). У тебя 75 клеток.")
print("   • Нанимай 5–7 рабочих (они дёшевы относительно дохода).")
print("   • Добавляй коров ($600) и овец ($500).")
print("   • CARE каждый день — даёт ×2-×4 к продукту.")
print("   • Собирай удобрение (FERTILIZER) — используй на культурах.")
print()
print("🔴 ДНИ 25–30: ПРОДАЖА И ЗАКРЫТИЕ")
print("   • НЕ покупай новых животных — не успеют окупиться.")
print("   • НЕ расширяйся — SE ($4000) не отобьётся.")
print("   • Продавай ВСЁ накопленное, но понемногу (не обваливай цену).")
print("   • Последние 2-3 дня: активно сливай остатки.")
print("   • На 718-м ходу — ПОСЛЕДНЯЯ продажа (после неё будет PASS).")
print()
print("=" * 100)
print("🎯 ЧТО РЕШАЕТ ПОБЕДУ")
print("=" * 100)
print()
print("1. Что ты посадил: пшеница — стабильность, дыня — максимум с клетки,")
print("   животные — постоянный доход, но требуют корма и внимания.")
print()
print("2. Когда продал: не слить всё сразу. Цена падает от каждой продажи.")
print("   Продавай в день, когда продукт в дефиците (город его съел).")
print()
print("3. Как использовал рабочих: каждый рабочий = 24 действия в день.")
print("   Простой рабочего = потерянные деньги на его зарплату.")
print()
print("4. Расширение: покупай землю только когда есть чем засадить")
print("   и есть рабочие, которые эту землю обработают.")
print()
print("=" * 100)
print("💡 ЧАСТЫЕ ОШИБКИ НОВИЧКОВ")
print("=" * 100)
print("  ❌ Не полил в день посадки → на утро сорняк (нет грейс-периода!)")
print("  ❌ Слил весь урожай в один ход → цена обвалилась до $1")
print("  ❌ Купил коров/овец в день 25 → не успели окупиться")
print("  ❌ Нанял 10 рабочих на 6 клеток → они простаивают, зарплата уходит в трубу")
print("  ❌ Не покормил животное 2 дня → сбежало, потерял $300-600")
print("  ❌ Переполнил сарай (>100) → лишнее уничтожилось ночью")
print("  ❌ Забыл про 718-й ход → продукты остались в сарае, не продались")